In [48]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['figure.figsize'] = 12,6
plt.rcParams['font.size'] = 14
plt.rcParams['axes.unicode_minus'] = False

# 데이터 전처리 관련 ####################################################
# 결측치 처리
from sklearn.impute import SimpleImputer
# 표준화
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
# 인코더
from sklearn.preprocessing import LabelEncoder
# 원핫 인코더
from sklearn.preprocessing import OneHotEncoder

# 학습 모델 성능 관련 ####################################################
# 원하는 비율로 데이터를 나누기 위해
from sklearn.model_selection import train_test_split
# K-Fold 교차 검증
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold
# 학습곡선
from sklearn.model_selection import learning_curve
# 하이퍼 파라미터 튜닝
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint
import optuna
from optuna.samplers import TPESampler # 데이터를 랜덤샘플링하기 위함
from optuna.pruners import MedianPruner
optuna.logging.set_verbosity(optuna.logging.WARNING) # 수치가 떨어지면 경고로그가 뜨는데 그거를 막아줌

# 모델 성능평가 #############################################
# 회귀용
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import r2_score
# 분류용
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score

from sklearn.metrics import roc_curve
from sklearn.metrics import auc
from sklearn.metrics import roc_auc_score
from sklearn.metrics import ConfusionMatrixDisplay

# 피처 선택 ################################################
from sklearn.feature_selection import VarianceThreshold
from sklearn.feature_selection import RFE
from sklearn.inspection import permutation_importance

# 학습모델 ##################################################
#분류
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.naive_bayes import GaussianNB
from catboost import CatBoostClassifier

#회귀
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.linear_model import ElasticNet
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import BayesianRidge
from catboost import CatBoostRegressor

# 결정트리를 시각화할 수 있는 라이브러리
from sklearn.tree import plot_tree

# 차원축소
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.manifold import TSNE

# 연관규칙 학습
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import fpgrowth
from mlxtend.frequent_patterns import association_rules

# 군집
from sklearn.cluster import KMeans
from sklearn.cluster import DBSCAN
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.mixture import GaussianMixture
from sklearn.cluster import MeanShift, estimate_bandwidth

# 파이프라인
from sklearn.pipeline import Pipeline

# KDE를 그리기 위한 통계값을 구할 수 있는 함수
from scipy.stats import gaussian_kde

# 피어슨 상관 계수 (연속형 수치형 데이터 vs 연속형 수치형 데이터)
from scipy.stats import pearsonr
# 카이제곱 검증 (범주형 데이터 vs 범주현 데이터, 순위 x)
from scipy.stats import chi2_contingency
# 스피어만 상관계수 (범주형 데이터 vs 범주형 데이터, 순위 O)
from scipy.stats import spearmanr
# 포인트 이분 상관계수 (범주형 데이터 vs 연속형 수치형 데이터)
from scipy.stats import pointbiserialr

# 객체를 파일에 저장
import pickle

# 불필요한 경고 뜨지 않게
import warnings
warnings.filterwarnings('ignore')

### 데이터 불러오기

In [2]:
train_df = pd.read_csv('data/titanic_train6.csv')
train_df

,Unnamed: 0,Survived,Sex,Parch,Fare,isChild,Pclass_1,Pclass_2,Pclass_3,Embarked_C,Title_Master,Title_Miss,Title_Mrs,FamilySizeGroup_0,FamilySizeGroup_2
0,0,0,1,0,2.110213,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,1,0,0,4.280593,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
2,2,1,0,0,2.188856,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
3,3,1,0,0,3.990834,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4,4,0,1,0,2.202765,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,886,0,1,0,2.639057,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
887,887,1,0,0,3.433987,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
888,888,0,0,2,3.196630,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
889,889,1,1,0,3.433987,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0


In [3]:
X = train_df.drop('Survived', axis=1)
y = train_df['Survived']

In [4]:
display(X)
display(y)

,Unnamed: 0,Sex,Parch,Fare,isChild,Pclass_1,Pclass_2,Pclass_3,Embarked_C,Title_Master,Title_Miss,Title_Mrs,FamilySizeGroup_0,FamilySizeGroup_2
0,0,1,0,2.110213,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,0,0,4.280593,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
2,2,0,0,2.188856,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
3,3,0,0,3.990834,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4,4,1,0,2.202765,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,886,1,0,2.639057,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
887,887,0,0,3.433987,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
888,888,0,2,3.196630,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
889,889,1,0,3.433987,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0


0      0
1      1
2      1
3      1
4      0
      ..
886    0
887    1
888    0
889    1
890    0
Name: Survived, Length: 891, dtype: int64

In [5]:
scaler1 = StandardScaler()
X_scaled = scaler1.fit_transform(X)
X_scaled

array([[-1.73010796,  0.73769513, -0.47367361, ..., -0.40583972,
        -1.2316449 , -0.2734756 ],
       [-1.72622007, -1.35557354, -0.47367361, ...,  2.4640269 ,
        -1.2316449 , -0.2734756 ],
       [-1.72233219, -1.35557354, -0.47367361, ..., -0.40583972,
         0.81192233, -0.2734756 ],
       ...,
       [ 1.72233219, -1.35557354,  2.00893337, ..., -0.40583972,
        -1.2316449 , -0.2734756 ],
       [ 1.72622007,  0.73769513, -0.47367361, ..., -0.40583972,
         0.81192233, -0.2734756 ],
       [ 1.73010796,  0.73769513, -0.47367361, ..., -0.40583972,
         0.81192233, -0.2734756 ]], shape=(891, 14))

In [6]:
# optuna는 목적함수를 만들어야함
# 목적함수
def objective(trial) : 
    # 하이퍼 파라미터 후보값 종류가 정해져 있는 것이라면
    solver = trial.suggest_categorical('solver',['libliner','lbfgs','saga','newton-cg'])
    # 하이퍼 파라미터 후보값이 정수 범위라면~
    max_iter = trial.suggest_int('max_iter', 100, 1000)
    # 하이퍼 파라미터 후보값이 실수 범위라면~
    C = trial.suggest_float('C', 1e-3, 1e2, log=True)
    
    params = {
        'solver' : 'saga',
        'max_iter' : max_iter,
        'C' : C
    }

    model = LogisticRegression(**params)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    score = cross_val_score(model, X_scaled, y, cv=cv, scoring='accuracy').mean()

    return score

# 튜닝 가동
sampler = TPESampler(seed=42)
study = optuna.create_study(direction='maximize', sampler=sampler, pruner=MedianPruner(n_warmup_steps=10))
study.optimize(objective, n_trials=50, show_progress_bar=True)

  0%|          | 0/50 [00:00<?, ?it/s]

In [7]:
study.best_value

0.8305191136777352

In [8]:
study.best_params

{'solver': 'libliner', 'max_iter': 435, 'C': 0.02634748223867888}

### KNN 하이퍼 파라미터 튜닝

In [9]:
def objective(trial) : 
    params = {
        # 이웃의 개수
        # 너무작으면 과대적합이 발생하고 너무 많으면 과소적합이 발생한다.
        'n_neighbors' : trial.suggest_int('n_neighbors',1,100),
        # 이웃의 투표 가중치 결정 방식
        # uniform : 모든 이웃 동일 가중치
        # distance : 가까운 이웃에게 더 높은 가중치 부여
        'weights' : trial.suggest_categorical('weights',['uniform','distance']),
        # 데이터 간의 거리를 측정하는 방식
        # euclidean : 두 점 사이의 최단 직선 거리
        # manhattan : 가로, 세로축을 따라 이동하는 격자 형태 거리(L1 distance)
        # minkowski : 유클리드와 맨하튼을 일반화한 거리 (p값에 따라 변환)
        # chebyshev : 각 좌표축의 차이 중 최대값을 거리로 정의 (체스판의 킹이 이동하는 방식)
        'metric' : trial.suggest_categorical('metric', ['euclidean','manhattan', 'minkowski', 'chebyshev']),
        # 가장 가까운 이웃을 찾기 위한 내부 계산 알고리즘
        # brute : 모든 데이터와 일일이 거리를 계산하는 단순 무식한 방식 (데이터가 적을 떄 정확)
        # kd_tree : 데이터를 다차원 공간에서 이진 트리 구조로 나누어 탐색(저차원 데이터에 최적)
        # ball_tree : 데이터를 다차원 구(원)형태로 묶어 계층적 탐색(고차원 데이터에 유리)
        # auto : 입력된 학습 데이터의 크기와 특성을 분석하여 위 방식 중 가장 적절한 것을 자동 선택(Default 값)
        # 보통 auto를 써서 아래 코드를 안해도 되지만~ 학습을 위해 아래 코드 실행
        'algorithm' : trial.suggest_categorical('algorithm', ['auto','ball_tree', 'kd_tree', 'brute']),
        # BallTree또는 KDTree 사용시 노드의 최소 크기
        # 트리의 깊이와 탐색 속도, 메모리 사용량에 영향을 줌(일반적으로 10~60사이 탐색)
        'leaf_size' : trial.suggest_int('leaf_size', 10, 60),
        # 모든 코어를 이용해 병렬 처리 할 수 있도록 설정한다.
        'n_jobs' : -1
    }

    # 만약 metric이 minkowski라면
    if params['metric'] == 'minkowski' :
        # p=1 : 맨하튼 거리와 동일  p=2 : 유클리드 거리와 동일
        # p가 커질수록 변화량이 줄어드르모, 작은 값 근처를 더 세밀하게 탐색
        params['p'] = trial.suggest_float('p',1.0,5.0)

    # 모델 생성
    model = KNeighborsClassifier(**params)

    # 교차검증
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    score = cross_val_score(model, X_scaled, y , cv=cv, scoring='accuracy').mean()

    return score

# 튜닝 시작
sampler = TPESampler(seed=42)
study = optuna.create_study(direction='maximize', sampler=sampler, pruner=MedianPruner(n_warmup_steps=10))
study.optimize(objective, n_trials=50, show_progress_bar=True)

knn_params = study.best_params.copy()
knn_score = study.best_value

print(f'최적 파라미터 : {knn_params}')
print(f'최고 정확도 : {knn_score}')

  0%|          | 0/50 [00:00<?, ?it/s]

최적 파라미터 : {'n_neighbors': 31, 'weights': 'distance', 'metric': 'euclidean', 'algorithm': 'brute', 'leaf_size': 51}
최고 정확도 : 0.838390559286925


### LogisticRegression 하이퍼 파라미터 튜닝

In [10]:
def objective(trial) : 
    # 하이퍼 파라미터
    params = {
        # 비용 함수를 최소화할 최적화 알고리즘 선택 
        # liblinear : 작은 데이터셋에 적합하며 L1, L2 규제 모두 지원
        # lbfgs : 기본값, 중대형 데이터셋에 적합(L2만 지원)
        # newton-cg : 뉴턴 방법, 정교하지만 연산량이 매우 많다(L2만 지원)
        # saga : 매우 큰 데이터셋에 최적화되어 있으며, 모든 규제 (L1,L2,ElasticNet)을 지원
        'solver' : trial.suggest_categorical('solver', ['liblinear', 'lbfgs', 'saga', 'newton-cg']),
        # 규제 강도의 역수
        # 작을 수록 규제가 강해짐
        'C' : trial.suggest_float('C', 1e-3, 1e2, log=True),
        # 알고리즘이 해를 찾을 때까지 반복할 최대 횟수
        'max_iter' : trial.suggest_int('max_iter', 100, 1000),
        # 랜덤시드
        'random_state' : 42
    }

    # 규제함수 설정
    if params['solver'] == 'liblinear' : 
        params['penalty'] = trial.suggest_categorical('penalty_liblinear', ['l1','l2'])
    elif params['solver'] == 'saga' :
        params['penalty'] = trial.suggest_categorical('penalty_saga', ['l1','l2','elasticnet', None])
    else :
        params['penalty'] = trial.suggest_categorical('penalty_lbfgs_newton', ['l2', None])

    # L1_ratio 비율 설정
    if params['penalty'] == 'elasticnet' :
        # 0에 가까우면 L2, 1에 가까우면 L1 규제와 비슷해진다
        params['l1_ratio'] = trial.suggest_float('l1_ratio',0,1)

    model = LogisticRegression(**params)

    cv = StratifiedKFold(n_splits=50, shuffle=True, random_state=42)
    score = cross_val_score(model, X_scaled, y, cv=cv, scoring='accuracy').mean()
    
    return score

sampler = TPESampler(seed=42)
study = optuna.create_study(direction='maximize', sampler=sampler, pruner=MedianPruner(n_warmup_steps=10))
study.optimize(objective, n_trials=5, show_progress_bar=True)

lg_params = study.best_params.copy()
lg_score = study.best_value

# 모델의 하이퍼 파라미터 이름으로 변경해주는 작업
for key in ['penalty_liblinear', 'penalty_saga','penalty_lbfgs_newton'] :
    if key in lg_params :
        lg_params['penalty'] = lg_params.pop(key)

print(f'최적 파라미터 : {lg_params}')
print(f'최고 정확도 : {lg_score}')

  0%|          | 0/5 [00:00<?, ?it/s]

최적 파라미터 : {'solver': 'lbfgs', 'C': 0.006026889128682512, 'max_iter': 240, 'penalty': None}
최고 정확도 : 0.8260784313725489


### SVC 하이퍼 파라미터 튜닝

In [11]:
def objective(trial) :
    params = {
        # 데이터를 고차원으로 매핑하는 공식
        # Linear : 선형 분리가 가능할 때 사용
        # poly : 다항식 커널, 곡선 형태의 경계를 만듬
        # rbf : 가우시안 커널, 기본값이자 가장 성능이 범용적임(일반적임)
        # sigmoid : 로지스틱 함수와 유사한 경계 생성
        'kernel' : trial.suggest_categorical('kernel', ['linear', 'poly','rbf','sigmoid']),
        # 오차에 대한 허용치 (규제강도)
        # 작을수록 마진이 커지고(과소적합 위험), 클수록 오차를 엄격하게 재한한다(과대적합 위험)
        'C' : trial.suggest_float('C', 1e-3,1e2,log=True),
        # 확률 기반 예측을 위해 활성화 (좀 더 정확한 예측을 위해서~)
        # 하이퍼 파라미터 때 하면 너무 시간이 오래걸린다.
        # 여기서는 일단 사용하지 않고 나중에 최종 학습할 때 사용한다.
        #'probability' : True,
        'random_state' : 42
    }

    # 결정 경계의 유연함(곡률)을 설정 (rbf, poly, sigmoid 에서 사용)
    # scale, auto는 데이터 분산을 활용한 자동 설정
    if params['kernel'] in ['rbf', 'poly', 'sigmoid'] : 
        params['gamma'] = trial.suggest_categorical('gamma',['scale', 'auto', 0.001,0.01,0.1,1])

    # 다항식 커널의 차수 설정(poly에서 사용)
    if params['kernel'] == 'poly' :
        params['degree'] = trial.suggest_int('degree',2,5)

    # 커널의 독립 상숫값(모델의 유연성 조절, poly와 sigmoid에서 사용)
    if params['kernel'] in ['poly', 'sigmoid'] :
        params['coef0'] = trial.suggest_float('coef0', 0.0, 10.0)

    model = SVC(**params)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    score = cross_val_score(model, X_scaled, y, cv=cv, scoring='accuracy').mean()
    return score

sampler = TPESampler(seed=42)
study = optuna.create_study(direction='maximize', sampler=sampler, pruner=MedianPruner(n_warmup_steps=10))
study.optimize(objective, n_trials=50, show_progress_bar=True)

svc_params = study.best_params.copy()
svc_score = study.best_value

print(f'최적 파라미터 : {study.best_params}')
print(f'최고 정확도 : {study.best_value}')
print(f'최고 정확도 : {svc_score}')

  0%|          | 0/50 [00:00<?, ?it/s]

최적 파라미터 : {'kernel': 'poly', 'C': 0.15594220755768762, 'gamma': 0.01, 'degree': 4, 'coef0': 3.4763343049658255}
최고 정확도 : 0.8350134957002071
최고 정확도 : 0.8350134957002071


### DecisionTree 하이퍼 파라미터 튜닝 

In [12]:
def objective(trial) :
    params = {
        # 분할 품질 측정하는 함수
        # gini : 지니 불순도. 계산이 빠름
        # entropy : 정보 이득 활용, 조금 더 정교
        'criterion' : trial.suggest_categorical('criterion', ['gini', 'entropy']),
        # 각 노드에서 분할을 선택하는 전략
        # best : 최선 분할 선택
        # random : 무작위 분할 중 가장 좋은 것을 선택(과적합 방지 효과)
        'splitter' : trial.suggest_categorical('splitter', ['best','random']),
        # 트리의 최대 깊이(과적합 방지의 핵심 파라미터)
        # 너무 깊으면 과대적합이 발생, 너무 낮으면 과소적합이 발생
        'max_depth' : trial.suggest_int('max_depth',2,32),
        # 노드를 분할하기 위해 필요한 최소 샘플 수
        'min_samples_split' : trial.suggest_int('min_samples_split', 2,20),
        # 리프 노드(마지막)가 되기 위해 필요한 최소 샘플 수
        'min_samples_leaf' : trial.suggest_int('min_samples_leaf', 1,20),
        # 최적의 분할을 찾기 위해 고려할 피쳐의 수
        'max_features' : trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        'random_state' : 42
    }

    model = DecisionTreeClassifier(**params)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    score = cross_val_score(model, X_scaled, y, cv=cv, scoring='accuracy').mean()
    
    return score

sampler = TPESampler(seed=42)
study = optuna.create_study(direction='maximize', sampler=sampler, pruner=MedianPruner(n_warmup_steps=10))
study.optimize(objective, n_trials=50, show_progress_bar=True)

tree_params = study.best_params.copy()
tree_score = study.best_value

print(f'최적 파라미터 : {tree_params}')
print(f'최고 정확도 : {study.best_value}')
print(f'최고 정확도 : {tree_score}')

  0%|          | 0/50 [00:00<?, ?it/s]

최적 파라미터 : {'criterion': 'gini', 'splitter': 'random', 'max_depth': 23, 'min_samples_split': 9, 'min_samples_leaf': 7, 'max_features': None}
최고 정확도 : 0.8248885820099178
최고 정확도 : 0.8248885820099178


### Random Forest 하이퍼 파라미터 튜닝

In [13]:
def objective(trial) :
    params = {
        # 결정 트리의 개수
        # 많을 수록 안정적이지만 시간이 오래걸린다
        'n_estimators' : trial.suggest_int('n_estimators', 50,500),
        # 분할 품질 측정하는 함수
        # gini : 지니 불순도. 계산이 빠름
        # entropy : 정보 이득 활용, 조금 더 정교
        'criterion' : trial.suggest_categorical('criterion', ['gini', 'entropy']),
        # 트리의 최대 깊이(과적합 방지의 핵심 파라미터)
        # 너무 깊으면 과대적합이 발생, 너무 낮으면 과소적합이 발생
        'max_depth' : trial.suggest_int('max_depth',2,32),
        # 노드를 분할하기 위해 필요한 최소 샘플 수
        'min_samples_split' : trial.suggest_int('min_samples_split', 2,20),
        # 리프 노드(마지막)가 되기 위해 필요한 최소 샘플 수
        'min_samples_leaf' : trial.suggest_int('min_samples_leaf', 1,20),
        # 최적의 분할을 찾기 위해 고려할 피쳐의 수
        'max_features' : trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        # 데이터 샘플링 시 중복 허용 여부
        # True일 경우 배깅 방식으로 학습 데이터의 일부를 무작위로 추출함
        'bootstrap' : trial.suggest_categorical('bootstrap', [True,False]),
        # CPU 코어를 모두 사용하도록~
        'n_jobs' : -1,
        # 랜덤 시드
        'random_state' : 42
    }

    model = RandomForestClassifier(**params)
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    score = cross_val_score(model, X_scaled,y, cv=cv, scoring='accuracy').mean()
    return score

sampler = TPESampler(seed=42)
study = optuna.create_study(direction='maximize', sampler=sampler, pruner=MedianPruner(n_warmup_steps=10))
study.optimize(objective, n_trials=5, show_progress_bar=True)

rf_params = study.best_params.copy()
rf_score = study.best_value

print(f'최적 파라미터 : {rf_params}')
print(f'최고 정확도 : {rf_score}')

  0%|          | 0/5 [00:00<?, ?it/s]

최적 파라미터 : {'n_estimators': 218, 'criterion': 'gini', 'max_depth': 20, 'min_samples_split': 4, 'min_samples_leaf': 4, 'max_features': 'log2', 'bootstrap': True}
최고 정확도 : 0.8338899001945892


In [14]:
model = RandomForestClassifier(**rf_params)

### GradientBoosting 하이퍼 파라미터 튜닝

In [15]:
def objective(trial) :
    params = {
        # 결정 트리의 개수
        # 많을 수록 안정적이지만 시간이 오래걸린다
        'n_estimators' : trial.suggest_int('n_estimators', 50,500),
        # 학습률
        # 각 트리의 기여도를 조절하는 학습률
        # 값이 작을수록 정교한 학습이 가능하지만 더 많은 트리가 필요하다.
        'learning_rate' : trial.suggest_float('learning_rate', 0.001, 0.5, log=True),
        # 트리의 최대 깊이(과적합 방지의 핵심 파라미터)
        # 너무 깊으면 과대적합이 발생, 너무 낮으면 과소적합이 발생
        'max_depth' : trial.suggest_int('max_depth',2,32),
        # 노드를 분할하기 위해 필요한 최소 샘플 수
        'min_samples_split' : trial.suggest_int('min_samples_split', 2,20),
        # 리프 노드(마지막)가 되기 위해 필요한 최소 샘플 수
        'min_samples_leaf' : trial.suggest_int('min_samples_leaf', 1,20),
        # 최적의 분할을 찾기 위해 고려할 피쳐의 수
        'max_features' : trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        # 각 트리를 학습 시킬 때 사용할 데이터의 비율
        # 1.0 보다 작으면 과적합 방지에 효과가 있다.
        'subsample' : trial.suggest_float('subsample', 0.5,1.0),
        # 랜덤 시드
        'random_state' : 42
    }
    model = GradientBoostingClassifier(**params)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    score = cross_val_score(model, X_scaled, y, cv=cv, scoring='accuracy').mean()
    return score

sampler = TPESampler(seed=42)
study = optuna.create_study(direction='maximize', sampler=sampler, pruner=MedianPruner(n_warmup_steps=10))
study.optimize(objective, n_trials=50, show_progress_bar=True)

gb_params = study.best_params
gb_score = study.best_value

print(f'최적의 파라미터 : {gb_params}')
print(f'최고의 정확도 : {gb_score}')

  0%|          | 0/50 [00:00<?, ?it/s]

최적의 파라미터 : {'n_estimators': 127, 'learning_rate': 0.03022306289342994, 'max_depth': 11, 'min_samples_split': 9, 'min_samples_leaf': 9, 'max_features': None, 'subsample': 0.53324014613012}
최고의 정확도 : 0.8495762977841942


In [16]:
model = GradientBoostingClassifier(**gb_params)

### LightGBM 하이퍼파라미터 튜닝

In [18]:
def objective(trial) :
    params = {
        # 결정 트리의 개수
        # 많을 수록 안정적이지만 시간이 오래걸린다
        'n_estimators' : trial.suggest_int('n_estimators', 50,500),
        # 학습률
        # 각 트리의 기여도를 조절하는 학습률
        # 값이 작을수록 정교한 학습이 가능하지만 더 많은 트리가 필요하다.
        'learning_rate' : trial.suggest_float('learning_rate', 0.001, 0.5, log=True),
        # 트리가 가질 수 있는 최대 리프 노드 개수 (★LGBM의 핵심 파라미터★)
        # 모델의 복잡도를 결정하며 너무 크면 과적합이 발생한다.
        'num_leaves' : trial.suggest_int('num_leaves', 2, 256),
        # 트리의 최대 깊이(과적합 방지의 핵심 파라미터)
        # 너무 깊으면 과대적합이 발생, 너무 낮으면 과소적합이 발생
        'max_depth' : trial.suggest_int('max_depth',2,32),
        # 리프노드가 되기 위해 필요한 최소 데이터 수
        # 과적합을 방지하는 주요 파라미터
        'min_child_samples' : trial.suggest_int('min_child_samples', 5, 100),
        # 각 트리를 학습 시킬 때 사용할 데이터의 비율 (행의 비율)
        # 1.0 보다 작으면 과적합 방지에 효과가 있다.
        'subsample' : trial.suggest_float('subsample', 0.5,1.0),
        # 각 트리를 학습 시킬 때 사용할 데이터 비율 (컬럼의 비율)
        # 1.0 보다 작으면 과적합 방지에 효과가 있다.
        'colsample_bytree' : trial.suggest_float('colsample_bytree',0.5,1.0),
        # L1 규제 강도
        'reg_alpha' : trial.suggest_float('reg_alpha', 1e-8,10.0, log=True),
        # L2 규제 강도
        'reg_lambda' : trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        # subsample를 활성화 하기 위해 1로 설정해야 한다.
        'subsample_freq' : 1,
        # 랜덤시드
        'random_state' : 42,
        # CPU 최대한 활용
        'n_jobs' : -1,
        # 피쳐 중요도 계산 방식
        'importance_type' : 'gain',
        # 로그 메시지 보이지 않게 설정
        'verbosity' : -1   
    }

    model = LGBMClassifier(**params)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    score = cross_val_score(model, X_scaled, y, cv=cv, scoring='accuracy').mean()

    return score

sampler = TPESampler(seed=42)
study = optuna.create_study(direction='maximize', sampler=sampler, pruner=MedianPruner(n_warmup_steps=10))
study.optimize(objective, n_trials=50, show_progress_bar=True)

lgb_params = study.best_params.copy()
lgb_score = study.best_value

print(f'최적 파라미터 : {lgb_params}')
print(f'최고 정확도 : {lgb_score}')
    

  0%|          | 0/50 [00:00<?, ?it/s]

최적 파라미터 : {'n_estimators': 335, 'learning_rate': 0.006923563846005429, 'num_leaves': 113, 'max_depth': 7, 'min_child_samples': 28, 'subsample': 0.9878553132174219, 'colsample_bytree': 0.9652076670570284, 'reg_alpha': 0.011010644111227453, 'reg_lambda': 1.912414341419089e-08}
최고 정확도 : 0.8439771514656957


In [19]:
model = LGBMClassifier(**lgb_params)

### XGBoost 하이퍼 파라미터 튜닝

In [21]:
def objective(trial) :
    params = {
        # 결정 트리의 개수
        # 많을 수록 안정적이지만 시간이 오래걸린다
        'n_estimators' : trial.suggest_int('n_estimators', 50,500),
        # 학습률
        # 각 트리의 기여도를 조절하는 학습률
        # 값이 작을수록 정교한 학습이 가능하지만 더 많은 트리가 필요하다.
        'learning_rate' : trial.suggest_float('learning_rate', 0.001, 0.5, log=True),
        # 트리의 최대 깊이(과적합 방지의 핵심 파라미터)
        # 너무 깊으면 과대적합이 발생, 너무 낮으면 과소적합이 발생
        # ★XGBoost는 깊이 제어가 매우 중요하다★
        'max_depth' : trial.suggest_int('max_depth',2,32),
        # 자식 노드에 필요한 모든 관측치의 최소 가중치 합
        # 이 값이 클수록 더 보수적인 모델이 되어 과적합을 방지한다
        'min_child_weight' : trial.suggest_int('min_child_weight',1,10),
        # 트리 분할을 위해 필요한 최소 손실 감소 값
        # 값이 클수록 트리가 보수적으로 성장한다.
        'gamma' : trial.suggest_float('gamma', 1e-8, 1.0, log=True),
        # 각 트리를 학습 시킬 때 사용할 데이터의 비율 (행의 비율)
        # 1.0 보다 작으면 과적합 방지에 효과가 있다.
        'subsample' : trial.suggest_float('subsample', 0.5,1.0),
        # 각 트리를 학습 시킬 때 사용할 데이터 비율 (컬럼의 비율)
        # 1.0 보다 작으면 과적합 방지에 효과가 있다.
        'colsample_bytree' : trial.suggest_float('colsample_bytree',0.5,1.0),
        # L1 규제 강도
        'reg_alpha' : trial.suggest_float('reg_alpha', 1e-8,10.0, log=True),
        # L2 규제 강도
        'reg_lambda' : trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        # 랜덤시드
        'random_state' : 42,
        # GPU 사용 설정
        'tree_method' : 'hist',
        'device' : 'cuda',
        'n_jobs' : -1,
        # 아래는 권장으로 향후에 디폴트로 하이퍼파라미터에서 빠질 수 있음
        'use_label_encoder' : False,
        'eval_metric' : 'logloss'
    }

    model = XGBClassifier(**params)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    score = cross_val_score(model, X_scaled, y , cv=cv, scoring='accuracy').mean()
    return score

sampler = TPESampler(seed=42)
study = optuna.create_study(direction='maximize', sampler=sampler, pruner=MedianPruner(n_warmup_steps=10))
study.optimize(objective, n_trials=50, show_progress_bar=True)

xgb_params = study.best_params.copy()
xgb_score = study.best_value

print(f'최적 파라미터 : {xgb_params}')
print(f'최고 정확도 : {xgb_score}')

  0%|          | 0/50 [00:00<?, ?it/s]

최적 파라미터 : {'n_estimators': 481, 'learning_rate': 0.03929616825914977, 'max_depth': 9, 'min_child_weight': 4, 'gamma': 0.00010477308733493709, 'subsample': 0.9259876956330169, 'colsample_bytree': 0.8271882613853765, 'reg_alpha': 0.0037022205646195516, 'reg_lambda': 2.2286349474617573e-05}
최고 정확도 : 0.8461866800577491


In [22]:
model = XGBClassifier(**xgb_params)

### 나이브베이지 하이퍼 파라미터 튜닝

In [25]:
def objective(trial) :
    params = {
        # 분산에 더해지는 가장 큰 분산의 비율
        # 계산 과정에서 분모가 0이 되는 것을 방지하고, 곡선을 부드럽게 만들어
        # 데이터가 정규분포를 완벽히 따르지 않을 때 모델을 안정화한다.
        'var_smoothing' : trial.suggest_float('var_smoothing',1e-9,1e-1,log=True)
    }

    model = GaussianNB(**params)
    cv= StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    score = cross_val_score(model,X_scaled,y, cv=cv, scoring='accuracy').mean()

    return score

sampler = TPESampler(seed=42)
study = optuna.create_study(direction='maximize', sampler=sampler, pruner=MedianPruner(n_warmup_steps=10))
study.optimize(objective, n_trials=50, show_progress_bar=True)

gn_params = study.best_params.copy()
gn_score = study.best_value

print(f'최적의 파라미터 : {gn_params}')
print(f'최고의 정확도 : {gn_score}')

  0%|          | 0/50 [00:00<?, ?it/s]

최적의 파라미터 : {'var_smoothing': 0.08467975927999691}
최고의 정확도 : 0.8114619295712762


In [27]:
model = GaussianNB(**gn_params)

### CatBoost 하이퍼 파라미터 튜닝

In [40]:
def objective(trial) :
    params = {
        # 부스팅 단계의 수(트리의 개수)
        'iterations' : trial.suggest_int('iterations',100,600),
        # 학습률
        # 각 트리의 기여도를 조절하는 학습률
        # 값이 작을수록 정교한 학습이 가능하지만 더 많은 트리가 필요하다.
        'learning_rate' : trial.suggest_float('learning_rate', 0.001, 0.5, log=True),
        # 트리의 깊이
        # Catboost는 GPU 사용시 깊이가 8을 넘어가면 연산량이 급증한다.
        'depth': trial.suggest_int('depth', 4, 8),
        # L2 규제 계수
        # 모델의 복잡도를 제어하여 과적합을 방지한다.
        'l2_leaf_reg' : trial.suggest_float('l2_leaf_reg',1e-1, 10.0, log=True),
        # 분할 선택시 무작위성을 부여하여 과적합 방지
        'random_strength' : trial.suggest_float('random_strength',1e-9,10.0, log=True),
        # 배깅의 강도조절
        # 값이 높을수록 더 공격적인 샘플링을 진행한다(각 트리가 학습하는 데이터의 중복 비율을 높여 과적합을 예방
        'bagging_temperature' : trial.suggest_float('bagging_temperature', 0.0,1.0),
        # 수치형 변수를 나타내는 최소 버킷의 개수
        # GPU 가속의 핵심
        'border_count' : trial.suggest_int('border_count', 32, 128),
        # 랜덤시드
        'random_state' : 42,
        # GPU 가속 설정
        'task_type' : 'GPU',
        # 사용할 GPU의 번호
        'devices' : '0',
        # GPU 사용시 bagging_temperature와 호환되는 타입
        'bootstrap_type' : 'Poisson',
        # GPU 메모리 사용 효율화(최대 80%까지 쓰겠다)
        'gpu_ram_part' : 0.8,
        # 로그 숨김
        'verbose' : False,
        # 로그 파일 생성 방지
        'allow_writing_files' : False
    }

    model = CatBoostClassifier(**params)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    score = cross_val_score(model, X_scaled,y, cv=cv,scoring='accuracy').mean()

    return score

sampler = TPESampler(seed=42)
study = optuna.create_study(direction='maximize', sampler=sampler, pruner=MedianPruner(n_warmup_steps=10))
study.optimize(objective, n_trials=50, show_progress_bar=True)

cb_params= study.best_params.copy()
cb_score = study.best_value

print(f'최적의 파라미터 : {cb_params}')
print(f'최고의 점수 : {cb_score}')

  0%|          | 0/50 [00:00<?, ?it/s]

최적의 파라미터 : {'iterations': 231, 'learning_rate': 0.024540855629982348, 'depth': 6, 'l2_leaf_reg': 3.6687632450568395, 'random_strength': 4.160749563672509e-07, 'bagging_temperature': 0.7243822488848977, 'border_count': 95}
최고의 점수 : 0.8305316678174627


In [41]:
model = CatBoostClassifier(**cb_params)

### 각 모델별 점수 비교

In [46]:
score_list = [
    knn_score, lg_score, svc_score, tree_score, rf_score, 
    gb_score, lgb_score, xgb_score, gn_score, cb_score
]

model_names = [
    'KNeighborsClassifier', 'LogisticRegression', 'SVC', 'DecisionTreeClassifier', 'RandomForestClassifier',
    'GradientBoostingClassifier', 'LGBMClassifier', 'XGBClassifier', 'GaussianNB', 'CatBoostClassifier'
]

score_df = pd.DataFrame(score_list, index=model_names, columns=['score'])
score_df.sort_values('score', ascending=False, inplace=True)
score_df

,score
GradientBoostingClassifier,0.849576
XGBClassifier,0.846187
LGBMClassifier,0.843977
KNeighborsClassifier,0.838391
SVC,0.835013
RandomForestClassifier,0.833890
CatBoostClassifier,0.830532
LogisticRegression,0.826078
DecisionTreeClassifier,0.824889
GaussianNB,0.811462


In [47]:
# 가장 성적이 좋은 모델을 학습시킨다.
best_model = GradientBoostingClassifier(**gb_params)
best_model.fit(X_scaled,y)

GradientBoostingClassifier(learning_rate=0.03022306289342994, max_depth=11,
                           min_samples_leaf=9, min_samples_split=9,
                           n_estimators=127, subsample=0.53324014613012)

In [49]:
# 모델 저장
with open('model/project1.dat','wb') as fp :
    pickle.dump(best_model, fp)
    pickle.dump(scaler1, fp)